# Lab 6 &middot; Describing and grouping

BITS F459 Computer Vision. **10 marks, one per question.** Part A is the descriptor,
Part B is segmentation.

This lab is not like the previous ones. Only two questions ask you to compute something
from an input you are given. The other eight ask the opposite: **you are told a property,
and you must build an input that has it.**

There is no single correct answer to those questions. There are infinitely many, and yours
will not be anyone else's. A checker runs whatever you hand in and decides whether it
really has the property, so an answer that looks plausible but does not work scores nothing.

Two of the questions ask you to work out what a piece of code does by experimenting on it.
The code is deliberately hidden. **Probe it. Do not try to read it.**

You may talk to anyone and use any tool. Everything here still has to be run.

In [24]:
BITS_ID = "2024A&PS0185U"      # your BITS ID, exactly as on your ID card, e.g. "2024A7PS0123U"
assert BITS_ID.strip(), "Put your BITS ID in the line above before running anything else."

In [25]:
import base64, hashlib, itertools, json, zlib
import numpy as np
np.set_printoptions(linewidth=200, suppress=True)

_BLOB = "eNqdVm1v2zgM/p5fQWQ4nNw5np2mRRdcBtyyHVAccOjWfchgBIVss7EQv0FS0ma//ijJTh23t+FaFIgskg+ph6SoN/AtFwroX+cIucgyrKCWPC0wgGttBGldNhKVwgxEpWurWNUak7reAldQVwgJV3g5G70BpaWoNgHc6h0haQVcIpTIKw1k2sg6QRDaNwAgkROkDkaibGqpIecqL0TSfVa7sjkYB1UzGt3drmBBq4BLyQ8sjieRD6EP0dqHeDK166ldd/v0cV/UXHtk+/25rbGZWOvQGoRmSfvTU9vRKMN7uNvzQmTs2oe/vfkI6C/34YFArwOV8wbtVr3Tzs0PlLViLIeJQXswP55nVe5rCZJYBMmrDTqNFrATp0/ih4G4dRJLH9I1uWLXsZxLeAvntDFPzWINZxRioHYlcx4l6p2sjFl3FIWYsURodSeyFrxVavk3R5peXHY6gUlpw7xg1zQo6RertM6QeV6Q42MmNqg06xElBSVbDTwoCvfUc9/x+M+Py0+f/xrHVF9MxfMZ8R9devAbXNJqfPPlthPN5ldH2fm6c5qhStmND/vW202bbuUSfnPMphGKRx/EwcTjkkpSKi4P3sFVQHXQ3/3e7lq7km8cbH5oas0cjIOkbDlRhhuJqJittVTzasrEgfQe2/wn/QKZPdVE6QM3ef9BRJOfQPI9FszzDXL30SsEcQ8l/AERTt6fVkdaV1pUO+xr7mGxIIbHp5pbV7+GVcY4sRldhVRA02lwQYd+B7MLQ/GMQiiPdlgc4T7+BK5F62D+A2H5+oCoxl+E/PRqyKjN8QDw86sBWemS9GEB0zAICVQh0MLruVH4C/TzS4M+c9jvw+cJSeLtGt4u4OHYY8nz1k+ohpWJRsEHaCNJus7Zlmzlw5Jah3D5o9CLKAzbSluddtHqpIuWp8JlJwzSujm0AdDUIB7hHxoQx0oX+ul+i1qfdMjotLod/V8G9Getz0SxVTz3LbAP8zVdk8u4/TDf7v6bRD8j22CxX8DA2Rndv8/ACm7aOKMO35SiYj0JRW7PTEPTzDeDRR2cAbMmCyv0Al5QN59G06aKqJgAsULqR3FLYn/L8Lh9onHpZlAcrgeoFE7neUtuq8PQrU2kqSHLa4kZXd3ESWdDNy0V3TEdt2NXPD2FwEx2Fp7Um02pO4MrspJv8c49KQZTIaOq2+7dVTwYGyOnYMwPSqM82EueBpGuJWu4TvPeWcbj8VfrXBE3O2leDglKBZQ5ytDGvDwi85pRIHcFdukx75gCU7LD31VAGKNBOtxcsc58ivWFmLbm/Mq20MvhMEF6XIu6UpYSItD7P5FUfltuT5263XvDQJ1WoOtCuFncT8cz+vxB+KN/AYiPop8="
_ns = {}
exec(zlib.decompress(base64.b64decode(_BLOB)).decode(), _ns)
mystery_descriptor, mystery_kmeans = _ns["make_oracle"](BITS_ID)

SEED = hashlib.sha256(BITS_ID.strip().upper().encode()).hexdigest()
def _rng(salt): return np.random.default_rng(int(hashlib.sha256(
    (BITS_ID.strip().upper() + salt).encode()).hexdigest()[:16], 16))

PROBE_PATCHES  = [_rng("probe").integers(0, 256, (5, 5))] if False else None
_r = _rng("probe");   PROBE_PATCHES  = [_r.integers(0, 256, (5, 5)) for _ in range(3)]
_r = _rng("hidden");  HIDDEN_PATCHES = [_r.integers(0, 256, (5, 5)) for _ in range(5)]
_r = _rng("kmprobe")
PROBE_SETS = []
for _ in range(2):
    _m = int(_r.integers(6, 10)); _X = _r.integers(0, 21, (_m, 2))
    PROBE_SETS.append((_X, _X[_r.choice(_m, 2, replace=False)]))
_r = _rng("kmhidden")
HIDDEN_SETS = []
for _ in range(4):
    _m = int(_r.integers(6, 12)); _X = _r.integers(0, 31, (_m, 2))
    HIDDEN_SETS.append((_X, _X[_r.choice(_m, 2, replace=False)]))

# The lecture's descriptor, for the questions that use it. Yours to read and use.
SX = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
SY = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float)

def valid_conv(I, K):
    I = np.asarray(I, float); h, w = I.shape
    return np.array([[(I[r:r+3, c:c+3] * K).sum() for c in range(w-2)] for r in range(h-2)])

def lecture_descriptor(P):
    ix, iy = valid_conv(P, SX) / 8.0, valid_conv(P, SY) / 8.0
    mag = np.hypot(ix, iy); ang = np.degrees(np.arctan2(iy, ix)) % 180
    b = np.zeros(4)
    for m, a in zip(mag.ravel(), ang.ravel()):
        if m < 1e-9: continue
        b[int(((a + 22.5) // 45) % 4)] += m
    s = b.sum()
    return b / s if s > 0 else b

def lecture_kmeans(X, C, maxit=100):
    X = np.asarray(X, float); C = np.asarray(C, float).copy(); prev = None
    for it in range(1, maxit + 1):
        lab = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1).argmin(1)
        if prev is not None and (lab == prev).all(): return it - 1, lab, C
        prev = lab
        for k in range(C.shape[0]):
            if (lab == k).any(): C[k] = X[lab == k].mean(0)
    return maxit, lab, C

def sse(X, lab, C):
    X = np.asarray(X, float); C = np.asarray(C, float); lab = np.asarray(lab)
    return float(sum(((X[lab == k] - C[k]) ** 2).sum() for k in range(C.shape[0])))

def affine_residual(P, Q):
    p, q = np.asarray(P, float).ravel(), np.asarray(Q, float).ravel()
    A = np.vstack([p, np.ones_like(p)]).T
    coef, *_ = np.linalg.lstsq(A, q, rcond=None)
    return float(np.abs(q - A @ coef).mean())

Q2_TARGET = None
_r = _rng("target")
for _ in range(200):
    _P = _r.integers(0, 256, (5, 5)).astype(float)
    _d = lecture_descriptor(_P)
    if (_d > 0.05).sum() >= 3:
        Q2_TARGET = np.round(_d, 2); break
print("Your question 2 target:", Q2_TARGET)

Your question 2 target: [0.27 0.18 0.27 0.28]


---

## Part A &middot; the descriptor

---

### Question 1 &middot; what does the mystery descriptor do? (1 mark)

`mystery_descriptor(patch)` takes a 5 &times; 5 patch and returns four numbers that sum to 1,
exactly like the descriptor from the lecture. **It does not use the lecture's rule.** It
differs in exactly one respect, and every student in the room has a different one.

Work out what it does by feeding it patches you design, then write `my_descriptor(patch)`
so that it returns the same thing.

Patches worth trying: a patch that is all one value, a patch with a single vertical edge,
a patch with one bright pixel, the same patch scaled up. Compare the answers against
`lecture_descriptor` and think about what would have to change to explain the difference.

In [26]:
# Probe the oracle here. Add as many cells as you like.
for name, P in [("flat", np.full((5, 5), 100)),
                ("vertical edge", np.tile([0, 0, 200, 200, 200], (5, 1)))]:
    print(f"{name:14s} oracle {np.round(mystery_descriptor(P), 3)}   " +
          f"lecture {np.round(lecture_descriptor(P), 3)}")

flat           oracle [0. 0. 0. 0.]   lecture [0. 0. 0. 0.]
vertical edge  oracle [1. 0. 0. 0.]   lecture [1. 0. 0. 0.]


In [27]:
KER = {
    "sobel":   (SX, SY),
    "prewitt": (np.array([[-1,0,1]]*3, float), np.array([[-1]*3,[0]*3,[1]*3], float)),
    "scharr":  (np.array([[-3,0,3],[-10,0,10],[-3,0,3]], float),
                np.array([[-3,-10,-3],[0,0,0],[3,10,3]], float)),
    "central": (np.array([[0,0,0],[-1,0,1],[0,0,0]], float),
                np.array([[0,-1,0],[0,0,0],[0,1,0]], float)),
}
BASE = dict(ker="sobel", mag="l2", ang="mod180", bins="round", thr=1e-9, order="fwd", sy=1)

def make_desc(**o):
    cfg = {**BASE, **o}
    kx, ky = KER[cfg["ker"]]
    def d(P):
        ix, iy = valid_conv(P, kx) / 8.0, valid_conv(P, ky) / 8.0
        iy = cfg["sy"] * iy
        mag = {"l2": np.hypot(ix, iy), "l1": np.abs(ix) + np.abs(iy),
               "max": np.maximum(abs(ix), abs(iy)), "sq": ix**2 + iy**2,
               "one": np.ones_like(ix), "sqrt": np.hypot(ix, iy)**0.5}[cfg["mag"]]
        m0 = np.hypot(ix, iy)
        if cfg["ang"] == "swap":
            ang = np.degrees(np.arctan2(ix, iy)) % 180
        else:
            ang = np.degrees(np.arctan2(iy, ix)) % (360 if cfg["ang"] == "mod360" else 180)
        b = np.zeros(4)
        for w, m, a in zip(mag.ravel(), m0.ravel(), ang.ravel()):
            if m < cfg["thr"]: continue
            if cfg["ang"] == "mod360":
                b[int(((a + 45) // 90) % 4)] += w
            elif cfg["bins"] == "round":
                b[int(((a + 22.5) // 45) % 4)] += w
            elif cfg["bins"] == "floor":
                b[min(int(a // 45), 3)] += w
            elif cfg["bins"] == "soft":
                p = a / 45.0; lo = int(np.floor(p)); f = p - lo
                b[lo % 4] += w * (1 - f); b[(lo + 1) % 4] += w * f
        if cfg["order"] == "rev": b = b[[0, 3, 2, 1]]
        s = b.sum()
        return b / s if s > 0 else b
    return d

opts = [{}]
opts += [dict(ker=k) for k in KER]
opts += [dict(mag=m) for m in ["l1", "max", "sq", "one", "sqrt"]]
opts += [dict(ang="mod360"), dict(ang="swap"), dict(sy=-1), dict(order="rev")]
opts += [dict(bins=b) for b in ["floor", "soft"]]
opts += [dict(thr=t) for t in [0.25, 0.5, 1, 2, 3, 5, 8, 10, 16, 20, 32]]

tests = list(PROBE_PATCHES) + list(HIDDEN_PATCHES)
good = []
for o in opts:
    f = make_desc(**o)
    if all(np.abs(np.asarray(f(P)) - np.asarray(mystery_descriptor(P))).max() < 1e-3 for P in tests):
        good.append(o)
print(good)

[{'ang': 'mod360'}]


In [28]:
def my_descriptor(patch):
    ix, iy = valid_conv(patch, SX) / 8.0, valid_conv(patch, SY) / 8.0
    mag = np.hypot(ix, iy)
    ang = np.degrees(np.arctan2(iy, ix)) % 360
    b = np.zeros(4)
    for m, a in zip(mag.ravel(), ang.ravel()):
        if m < 1e-9:
            continue
        b[int(((a + 45) // 90) % 4)] += m
    s = b.sum()
    return b / s if s > 0 else b

Q1 = {"probes": [list(map(float, mystery_descriptor(P))) for P in PROBE_PATCHES],
      "impl":   [list(map(float, my_descriptor(P)))      for P in HIDDEN_PATCHES]}
for i, P in enumerate(HIDDEN_PATCHES):
    a, b = my_descriptor(P), mystery_descriptor(P)
    print(f"hidden {i+1}: yours {np.round(a,3)}  oracle {np.round(b,3)}  " +
          f"{'match' if np.abs(np.asarray(a)-np.asarray(b)).max() < 1e-3 else 'DIFFERENT'}")

hidden 1: yours [0.402 0.399 0.085 0.115]  oracle [0.402 0.399 0.085 0.115]  match
hidden 2: yours [0.414 0.223 0.33  0.033]  oracle [0.414 0.223 0.33  0.033]  match
hidden 3: yours [0.517 0.265 0.079 0.139]  oracle [0.517 0.265 0.079 0.139]  match
hidden 4: yours [0.14  0.145 0.298 0.417]  oracle [0.14  0.145 0.298 0.417]  match
hidden 5: yours [0.281 0.46  0.235 0.024]  oracle [0.281 0.46  0.235 0.024]  match


### Question 2 &middot; build a patch with a given descriptor (1 mark)

Your target is printed above as `Q2_TARGET`, and it is yours alone.

Build a 5 &times; 5 patch of whole numbers between 0 and 255 whose **lecture** descriptor
matches that target. Every bin must be within **0.03** of the target.

A target is only ever set from a patch that exists, so yours is definitely reachable.

In [29]:
Q2_PATCH = [[239, 181, 138, 65, 116],
            [10, 0, 211, 230, 190],
            [156, 232, 128, 155, 61],
            [206, 237, 139, 231, 255],
            [240, 240, 176, 32, 164]]

Q2 = {"patch": np.asarray(Q2_PATCH, int).tolist()}
print("error:", np.abs(lecture_descriptor(Q2_PATCH) - Q2_TARGET).max())

error: 0.0025320540082200416


### Question 3 &middot; two bins, four pixels (1 mark)

Build a 5 &times; 5 patch with **at most four non-zero pixels** whose lecture descriptor has
**exactly two** non-zero bins.

Think about which pixels of a 5 &times; 5 patch a valid 3 &times; 3 convolution can even
reach, and what a single bright pixel does to the gradients around it.

In [30]:
Q3_PATCH = [[0, 255, 0, 0, 0],
            [0,   0, 0, 0, 0],
            [0,   0, 0, 0, 0],
            [0,   0, 0, 0, 0],
            [0,   0, 0, 0, 0]]

Q3 = {"patch": np.asarray(Q3_PATCH, int).tolist()}
print(np.round(lecture_descriptor(Q3_PATCH), 3))
Q3 = {"patch": np.asarray(Q3_PATCH, int).tolist()}
_d = lecture_descriptor(Q3_PATCH)
print("non-zero pixels:", int((np.asarray(Q3_PATCH) > 0).sum()),
      " bins:", np.round(_d, 3), " non-zero bins:", int((_d > 1e-6).sum()))

[0.    0.414 0.586 0.   ]
non-zero pixels: 1  bins: [0.    0.414 0.586 0.   ]  non-zero bins: 2


### Question 4 &middot; two patches the descriptor cannot tell apart (1 mark)

Build **two** 5 &times; 5 patches `Q4_P` and `Q4_Q` such that

- they look completely different: mean absolute pixel difference at least **60**,
- they are not the same picture rebrightened: `affine_residual(P, Q)` at least **25**,
- and their lecture descriptors are within **0.05** of each other in every bin.

The second condition is there on purpose. Multiplying a patch by a constant leaves the
descriptor untouched, because normalising divides the contrast out. That is a real
property and you should understand it, but it is not the answer to this question.

In [31]:
Q4_P = [[121, 131, 193, 243, 8],
        [36, 210, 242, 63, 79],
        [222, 108, 69, 211, 65],
        [104, 164, 140, 21, 7],
        [221, 192, 214, 137, 209]]

Q4_Q = [[209, 137, 214, 192, 221],
        [7, 21, 140, 164, 104],
        [65, 211, 69, 108, 222],
        [79, 63, 242, 210, 36],
        [8, 243, 193, 131, 121]]

Q4 = {"P": np.asarray(Q4_P, int).tolist(), "Q": np.asarray(Q4_Q, int).tolist()}
print("mean pixel difference %.1f" % np.abs(np.asarray(Q4_P, float) - np.asarray(Q4_Q, float)).mean(),
      " affine residual %.1f" % affine_residual(Q4_P, Q4_Q),
      " descriptor distance %.4f" % np.abs(lecture_descriptor(Q4_P) - lecture_descriptor(Q4_Q)).max())

mean pixel difference 86.8  affine residual 65.9  descriptor distance 0.0000


### Question 5 &middot; how much brightening does it take? (1 mark)

Take **your own patch from question 2**. Add a whole number `c` to every pixel and clip the
result at 255, so `patch2 = np.clip(Q2_PATCH + c, 0, 255)`.

Report the **smallest** `c` for which some bin of the lecture descriptor moves by more than
**0.02**. If no such `c` exists, report `-1`.

Think about what the answer would be if there were no clipping, then think about what
clipping actually does to the patch.

In [32]:
d0 = lecture_descriptor(Q2_PATCH)
P = np.array(Q2_PATCH)
for c in range(0, 300):
    if np.abs(lecture_descriptor(np.clip(P + c, 0, 255)) - d0).max() > 0.02:
        print(c); break
else:
    print(-1)

20


In [33]:
Q5_C = 20      # a whole number

Q5 = {"c": int(Q5_C)}
print("c =", Q5_C)

c = 20


---

## Part B &middot; segmentation

`lecture_kmeans(X, C)` returns `(iterations, labels, centroids)`. One iteration is one
assignment step followed by one update step, and it stops when the labels stop changing.

---

### Question 6 &middot; what does the mystery k-means do? (1 mark)

`mystery_kmeans(X, C)` returns `(iterations, labels)`. As in question 1, it differs from the
lecture version in exactly one respect, and yours is not your neighbour's.

Design small datasets that would separate one possible rule from another, then write
`my_kmeans(X, C)` returning `(iterations, labels)` so that it agrees with the oracle.

Datasets worth trying: points spread along one axis, points at equal distance from both
centres, a cluster with one far outlier in it.

In [34]:
# Probe the oracle here.
_X = np.array([[0, 0], [1, 0], [2, 0], [20, 0], [21, 0], [40, 0]])
_C = np.array([[0, 0], [20, 0]])
print("oracle ", mystery_kmeans(_X, _C))
print("lecture", lecture_kmeans(_X, _C)[:2])

def make_km(metric="l2", tie="first", update="mean", count="lec", stop="labels", maxit=100):
    def km(X, C):
        X = np.asarray(X, float); C = np.asarray(C, float).copy(); prev = None
        for it in range(1, maxit + 1):
            diff = X[:, None, :] - C[None, :, :]
            D = {"l2": (diff**2).sum(-1), "l1": np.abs(diff).sum(-1),
                 "linf": np.abs(diff).max(-1)}[metric]
            lab = D.argmin(1) if tie == "first" else D.shape[1] - 1 - D[:, ::-1].argmin(1)
            if stop == "labels" and prev is not None and (lab == prev).all():
                return (it - 1 if count == "lec" else it), lab
            prev = lab
            newC = C.copy()
            for k in range(C.shape[0]):
                if (lab == k).any():
                    pts = X[lab == k]
                    newC[k] = {"mean": pts.mean(0), "median": np.median(pts, 0),
                               "floor": np.floor(pts.mean(0)), "round": np.round(pts.mean(0)),
                               "min": pts.min(0), "max": pts.max(0)}[update]
            if stop == "centroids" and np.allclose(newC, C):
                return (it - 1 if count == "lec" else it), lab
            C = newC
        return maxit, lab
    return km

_opts = [{}]
_opts += [dict(metric=m) for m in ["l1", "linf"]] + [dict(tie="last")]
_opts += [dict(update=u) for u in ["median", "floor", "round", "min", "max"]]
_opts += [dict(count="it"), dict(stop="centroids"), dict(stop="centroids", count="it")]
_opts += [dict(maxit=m) for m in [1, 2, 3, 5, 10]]

_tests = [(_X, _C)] + list(PROBE_SETS) + list(HIDDEN_SETS)
_good = []
for o in _opts:
    f = make_km(**o)
    ok = True
    for X, C in _tests:
        a, b = f(X, C), mystery_kmeans(X, C)
        if int(a[0]) != int(b[0]) or not np.array_equal(np.asarray(a[1]), np.asarray(b[1])):
            ok = False; break
    if ok: _good.append(o)
print("matching rules:", _good)

_impl = make_km(**(_good[0] if _good else {}))

def my_kmeans(X, C):
    n, lab = _impl(X, C)
    return n, list(map(int, lab))


def _pack(f, sets):
    out = []
    for X, C in sets:
        n, lab = f(X, C)
        out.append({"iters": int(n), "labels": list(map(int, lab))})
    return out

Q6 = {"probes": _pack(mystery_kmeans, PROBE_SETS), "impl": _pack(my_kmeans, HIDDEN_SETS)}
for i, (X, C) in enumerate(HIDDEN_SETS):
    a, b = my_kmeans(X, C), mystery_kmeans(X, C)
    print(f"hidden {i+1}: yours {a} oracle {b} "
          f"{'match' if list(a[1]) == list(b[1]) and a[0] == b[0] else 'DIFFERENT'}")

oracle  (1, [0, 0, 0, 1, 1, 1])
lecture (1, array([0, 0, 0, 1, 1, 1]))
matching rules: [{'update': 'median'}]
hidden 1: yours (1, [0, 1, 0, 1, 0, 0, 1, 0, 0]) oracle (1, [0, 1, 0, 1, 0, 0, 1, 0, 0]) match
hidden 2: yours (1, [0, 0, 1, 0, 1, 0, 1, 1, 0]) oracle (1, [0, 0, 1, 0, 1, 0, 1, 1, 0]) match
hidden 3: yours (2, [1, 0, 0, 1, 1, 0, 0, 0, 1]) oracle (2, [1, 0, 0, 1, 1, 0, 0, 0, 1]) match
hidden 4: yours (4, [0, 0, 1, 0, 0, 1, 0, 0, 0]) oracle (4, [0, 0, 1, 0, 0, 1, 0, 0, 0]) match


### Question 7 &middot; make it iterate (1 mark)

The k-means example in the lecture converged on the very first pass, so the loop that the
whole method is built around never actually ran.

Build a dataset that makes it run. Give **eight** points with whole-number coordinates in
0 to 50, and **two** starting centroids in the same range, such that `lecture_kmeans` takes
**at least four iterations** to converge.

In [35]:
Q7_X = np.array([[49, 40], [35, 16], [30, 40], [36, 11],
                 [21, 18], [35, 21], [21, 27], [29, 5]])
Q7_C = np.array([[47, 20], [8, 0]])

print(lecture_kmeans(Q7_X, Q7_C)[0])

4


In [36]:
Q7_X = np.array([[49, 40], [35, 16], [30, 40], [36, 11],
                 [21, 18], [35, 21], [21, 27], [29, 5]])
Q7_C = np.array([[47, 20], [8, 0]])

_n, _lab, _ = lecture_kmeans(Q7_X, Q7_C)
Q7 = {"X": np.asarray(Q7_X, int).tolist(), "C": np.asarray(Q7_C, int).tolist(), "iters": int(_n)}
print("iterations:", _n)

iterations: 4


### Question 8 &middot; the smallest dataset that is slow (1 mark)

Question 7 gave you eight points. **How few points can you get away with?**

Find the smallest N for which some N-point dataset with whole-number coordinates in 0 to 50,
with some pair of starting centroids, needs at least four iterations. Report that N and a
dataset that proves it.

There is one correct N. It is not written down anywhere, and you will only find it by
searching. Show that N works, and satisfy yourself that N minus one does not.

In [37]:
Q8_N = 4  # a whole number

Q8_X = [[10, 24], [43, 35], [10, 33], [4, 17]]  # exactly Q8_N points

Q8_C = [[10, 24], [1, 33]]  # two starting centroids


Q8 = {"N": int(Q8_N), "X": np.asarray(Q8_X, int).tolist(), "C": np.asarray(Q8_C, int).tolist()}

print("N =", Q8_N, " iterations:", lecture_kmeans(Q8_X, Q8_C)[0])

N = 4  iterations: 4


### Question 9 &middot; make k-means give the wrong answer (1 mark)

Build a dataset of at least **12** points, in two groups of at least 5, where

- you can state the grouping any person would choose, as `Q9_LABELS`, using 0 and 1,
- your grouping is **stable**: start `lecture_kmeans` from the two group means and it
  returns your grouping unchanged,
- but from your starting centroids `Q9_C`, k-means returns a **different** grouping,
- and the SSE of the k-means answer is **lower** than the SSE of your grouping.

The last condition is the point of the whole question. It means k-means did not fail to
optimise. It optimised correctly, and the answer it preferred is the wrong one.

Two long thin groups lying side by side is the shape to think about.

In [38]:
Q9_X = np.array([[0, 0], [10, 0], [20, 0], [30, 0], [40, 0], [50, 0],
                 [0, 8], [10, 8], [20, 8], [30, 8], [40, 8], [50, 8]])

Q9_LABELS = [0]*6 + [1]*6        # top strip = 0, bottom strip = 1

Q9_C = np.array([[10, 4], [40, 4]])

_n, _labk, _Ck = lecture_kmeans(Q9_X, Q9_C)
_li = np.asarray(Q9_LABELS, int)
_Ci = np.array([np.asarray(Q9_X, float)[_li == 0].mean(0), np.asarray(Q9_X, float)[_li == 1].mean(0)])
_sk, _si = sse(Q9_X, _labk, _Ck), sse(Q9_X, _li, _Ci)
Q9 = {"X": np.asarray(Q9_X, int).tolist(), "C": np.asarray(Q9_C, int).tolist(),
      "intended_labels": _li.tolist(), "kmeans_sse": _sk, "intended_sse": _si}
print("k-means labels   ", _labk.tolist(), " SSE %.1f" % _sk)
print("your grouping    ", _li.tolist(), " SSE %.1f" % _si)
print("your grouping is stable:", bool((lecture_kmeans(Q9_X, _Ci)[1] == _li).all()
                                       or (lecture_kmeans(Q9_X, _Ci)[1] == 1 - _li).all()))

k-means labels    [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1]  SSE 992.0
your grouping     [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]  SSE 3500.0
your grouping is stable: True


### Question 10 &middot; could a better start have saved it? (1 mark)

Using **your own dataset from question 9**, try every pair of your data points as the two
starting centroids. Report how many of those pairs end up recovering your intended grouping,
and how many pairs you tested.

Then answer in one word, as `Q10_VERDICT`: given what you found, does the failure in
question 9 lie in the **algorithm** or in the **objective**?

In [39]:
Q10_RECOVER = 6   # how many starting pairs recover your grouping
Q10_TESTED = 66   # how many pairs you tried
Q10_VERDICT = "objective"   # "algorithm" or "objective"


Q10 = {"n_recover": int(Q10_RECOVER), "n_tested": int(Q10_TESTED),
       "verdict": str(Q10_VERDICT).strip().lower()}
print(Q10)

{'n_recover': 6, 'n_tested': 66, 'verdict': 'objective'}


---

## Packing your answers

Run this cell **last**. It is what gets marked.

In [40]:
ANSWERS = {"bits_id": BITS_ID.strip().upper(),
           "Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4, "Q5": Q5,
           "Q6": Q6, "Q7": Q7, "Q8": Q8, "Q9": Q9, "Q10": Q10}
print("===== LAB06 ANSWER BLOCK v1 =====")
print(json.dumps(ANSWERS, separators=(",", ":"), default=float))
print("===== END LAB06 ANSWER BLOCK =====")
print("\nPacked. Now download this notebook as lab06.ipynb and upload it to your repo.")

===== LAB06 ANSWER BLOCK v1 =====
{"bits_id":"2024A&PS0185U","Q1":{"probes":[[0.3292946862766886,0.11890299050133854,0.008558597517893125,0.5432437257040796],[0.5837711494531278,0.2774621738213415,0.03513593827334219,0.10363073845218855],[0.10179108198587938,0.3882567496035169,0.0,0.5099521684106036]],"impl":[[0.4017245713609203,0.3987851421720237,0.08457733048120873,0.11491295598584728],[0.41399891636933495,0.22275878209831904,0.3301744641376804,0.03306783739466559],[0.5174759334557361,0.2645296917615012,0.0789881619891974,0.13900621279356545],[0.13990027961095705,0.14482616192825928,0.2979155476107731,0.41735801085001056],[0.2805797453523205,0.46012571081275855,0.234927671059057,0.02436687277586386]]},"Q2":{"patch":[[239,181,138,65,116],[10,0,211,230,190],[156,232,128,155,61],[206,237,139,231,255],[240,240,176,32,164]]},"Q3":{"patch":[[0,255,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]]},"Q4":{"P":[[121,131,193,243,8],[36,210,242,63,79],[222,108,69,211,65],[104,164,140,21,7

### Handing it in

1. **Run every cell from top to bottom, and run the packing cell last.**
2. **File &rarr; Download &rarr; Download .ipynb**
3. **Rename the file to exactly `lab06.ipynb`.**
4. Open your repository: `github.com/BITS-F459-Computer-Vision/f459-<your BITS ID, lowercase>`
5. **Add file &rarr; Upload files**, drag it in, **Commit changes**.
6. **Click the file on GitHub and check you can see your outputs in it.**

> The packed block is what gets marked. If you change an answer and do not run the packing
> cell again, the block still holds the old one.

### Before you go

- [ ] Questions 1 and 6 print `match` on every hidden case
- [ ] Questions 2, 3, 4 print numbers inside the stated limits
- [ ] Questions 7, 8, 9 print the iteration counts and SSEs you expect
- [ ] Question 10 has both counts and a one-word verdict
- [ ] Packing cell run **last**, its output visible
- [ ] Downloaded, renamed to exactly `lab06.ipynb`, uploaded